### **Section 1: The API Anatomy & Web Fundamentals**

To understand **FastAPI**, we first have to understand the environment it lives in. Every time you use an app or a website, a conversation is happening between two computers.

1. What is an API?<br>
An **API (Application Programming Interface)** is a set of defined rules that allow one software application to communicate with another. It acts as an interface—a point where two different systems meet and exchange data.<br>
In the context of web development, when we talk about APIs, we are usually talking about Web APIs. These allow a front-end (like a website or mobile app) to request data from a back-end (a database or a server).<br>
**FastAPI** is the tool we use to build that **Server** and define those **API rules**


2. Client vs. Server<br>
To understand how an API works, you must understand the two roles involved in every interaction:
* **The Client:** This is the party that requests information. It could be your web browser (Chrome/Safari), a mobile app on your phone, or even another script running on a different computer.
* **The Server:** This is the party that responds to the request. It sits on a computer somewhere, waiting for a client to ask for something. It processes the request, looks up data, and sends a response back.

3. The Conversation: Request and Response
The Client and Server talk in a cycle:
* **The Request:** The Client sends a message to the Server (e.g., "Give me the temperature for New York").
* **The Response:** The Server processes that message and sends a reply back (e.g., "It is 70°F").

4. What is FASTAPI?<br>
**FastAPI** is a modern, high-performance web framework for building APIs with Python. It is designed to be easy to use while being incredibly fast—comparable to frameworks in languages like Go or Node.js. It relies heavily on Python type hints to handle data validation and documentation automatically.

### **Section 2: The HTTP Protocol**

Now that we understand the Client-Server relationship, we need to look at the rules they use to talk.<br>
**HTTP** (Hypertext Transfer Protocol) is the language of the web. Think of it as the "grammar" that ensures both sides understand what is happening.

There are three main parts to every HTTP conversation:
1. HTTP Methods (The Verbs)<br>
When a Client sends a request, it uses a specific "Method" to tell the Server what action it wants to perform.<br>
In FastAPI, you will see these used as "decorators" (the `@app.get` style we'll see later).

| Method | Technical Action | Purpose |
| --- | --- | --- |
| **GET** | Read | Retrieve data (e.g., viewing a profile). |
| **POST** | Create | Submit new data (e.g., creating a new account). |
| **PUT** | Replace | Update an entire record with new information. |
| **PATCH** | Modify | Update only specific parts of a record (e.g., changing just a password). |
| **DELETE** | Remove | Delete data from the server. |

2. URL Structure (The Address)<br>
A URL is more than just a link; it's a map for the API. Let's break down this example:
`https://example.com/items/42?color=red`

* **Protocol (`https`)**: The secure way the data is sent.
* **Domain (`example.com`)**: The name of the server you are talking to.
* **Path (`/items/42`)**: The specific "resource" you want. In this case, item number 42.
* **Parameters (`?color=red`)**: Extra instructions or filters. Here, we only want the red version of item 42.

3. JSON (The Language of Data)<br>
**JSON** (JavaScript Object Notation) is how we package data so it's easy to read. It looks almost identical to a Python dictionary.

**Example JSON for a new user:**

```json
{
  "username": "coder_jack",
  "bio": "Learning FastAPI!",
  "is_premium": true
}

```

API servers love JSON because it's lightweight and works with every programming language.

### **Section 3: Synchronous vs. Asynchronous Programming**

This is the "Fast" in **FastAPI**. To understand why this framework is so powerful, we need to look at how Python handles "waiting" versus "doing."

1. The Synchronous World (Blocking)<br>
In traditional Python programming, code is **Synchronous**. This means the computer follows your instructions exactly in order, one line at a time. If a line of code is waiting for something—like a response from a database—the entire program "blocks" (stops moving).<br>
**Technical Example: The Blocking Way**<br>
Imagine a script that needs to "wait" for two different tasks.

In [1]:
import time

def task_one():
    print("Task 1: Starting (takes 3 seconds)...")
    time.sleep(3)  # This stops the entire program
    print("Task 1: Done!")

def task_two():
    print("Task 2: Starting (takes 2 seconds)...")
    time.sleep(2)  # Again, everything stops
    print("Task 2: Done!")

start = time.perf_counter()
task_one()
task_two()
end = time.perf_counter()

print(f"Total time: {end - start:.2f} seconds")

# Result: 
# This takes 5 seconds. Task 2 cannot start until Task 1 is completely finished. 
# The computer sits idle during those `time.sleep` calls.

Task 1: Starting (takes 3 seconds)...
Task 1: Done!
Task 2: Starting (takes 2 seconds)...
Task 2: Done!
Total time: 5.00 seconds


2. The Asynchronous World (Non-blocking)<br>
**Asynchronous** programming allows the computer to start a task and, while waiting for it to finish, move on to other work. This is handled by three core components:
* * The Event Loop<br>
The **Event Loop** is the central manager. It keeps track of all running tasks. When a task hits a "waiting" point, it tells the Event Loop, "I'm waiting now; go do other work." The loop then switches to the next available task.
* * `async def` (The Coroutine)<br>
In Python, we use `async def` to define a function as a **Coroutine**. Unlike a regular function, a coroutine doesn't run immediately when called; it is scheduled to be run by the Event Loop.
* * `await` (The Pause Button)<br>
The `await` keyword is only used inside `async` functions. It tells the Event Loop: "You can pause me here. I'm waiting for an external result (like a database or a timer). Go run other tasks until I'm ready."
**Technical Example: The FastAPI/Async Way**

In [2]:
import asyncio
import time

async def task_one():
    print("Task 1: Starting...")
    await asyncio.sleep(3)
    print("Task 1: Done!")

async def task_two():
    print("Task 2: Starting...")
    await asyncio.sleep(3)
    print("Task 2: Done!")

async def main():
    start = time.perf_counter()
    await asyncio.gather(task_one(), task_two())
    end = time.perf_counter()
    print(f"Total time: {end - start:.2f} seconds")

await main()

# Result: 
# This takes only 3 seconds.
# While Task 1 was "sleeping," the Event Loop used that idle time to start and finish Task 2.

Task 1: Starting...
Task 2: Starting...
Task 1: Done!
Task 2: Done!
Total time: 3.00 seconds


3. Why this matters for FastAPI<br>
When a user visits your API, your code often has to wait for a database or another website.
* In a **Sync** server, if User A is waiting for a slow database, User B has to wait behind them.
* In **FastAPI (Async)**, the server pauses User A's request, processes User B's request in the meantime, and then goes back to User A once the database responds.<br>
To test this concept: In the `async` code example above, what do you think would happen if we accidentally used the regular `time.sleep(3)` instead of `await asyncio.sleep(3)` inside the `async def task_one()` function?


### **Section 4: Modern Python Tools**

Before we write our first line of FastAPI code, we need to understand the two "engines" that make it work: **Python Type Hints** and **Pydantic**. These aren't just extra features; they are the reason FastAPI can automatically validate data and create documentation for you.
1. **Python Type Hints**<br>
In older versions of Python, variables were "dynamic," meaning Python didn't care if a variable held a number or a piece of text until the code actually ran. **Type Hints** allow us to explicitly state what kind of data a variable *should* hold.<br>
**The Syntax:**
Instead of `name = "Alice"`, we write:

In [3]:
name: str = "Alice"
age: int = 25
is_learning: bool = True

**Why this matters for FastAPI:**
When you tell FastAPI that a function expects an `age: int`, FastAPI will:

1. **Read** the incoming data from the web request.
2. **Check** if it is actually an integer.
3. **Convert** it automatically (e.g., changing the text "25" into the number `25`).
4. **Error out** immediately if the user sends "twenty-five" instead of a number, before your code even touches it.

2. **Pydantic Foundations**<br>
While Type Hints work for simple variables, we often need to handle complex data, like a User or a Product. For this, we use **Pydantic**.<br>
Pydantic allows you to define "Schemas" (blueprints) using Python classes. Think of it as a strict checklist for your data.
**Example of a Pydantic Model:**

In [4]:
from pydantic import BaseModel

class User(BaseModel):
    id: int
    username: str
    email: str
    is_active: bool = True  # This provides a default value

If a Client sends a JSON request to your API to create a user, Pydantic will look at that JSON and make sure every single field matches the type you defined.

**Let's look at a practical example**<br>
Imagine we are building an API for a **Library**. We want to define what a "Book" looks like so that we can't accidentally add a book without a title or with a negative number of pages.

In [5]:
from pydantic import BaseModel

class Book(BaseModel):
    title: str
    author: str
    pages: int
    is_available: bool = True

Now, suppose a user sends this data to our API:

```json
{
  "title": "The Great Gatsby",
  "author": "F. Scott Fitzgerald",
  "pages": "180"
}

```

Notice that `"180"` is inside quotes, which technically makes it a **string** (text), not an **integer** (number).